# 

In [9]:
import json
import numpy as np
import pandas as pd 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, LayerNormalization, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.stem import WordNetLemmatizer
import nltk
import random 
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
with open('C:\\Users\\Lenovo\\Downloads\\archive(2)\\intents.json', 'r') as file:
    data = json.load(file)

df = pd.DataFrame(data['intents'])    

In [12]:
dic = {"tag":[], "patterns":[], "responses":[]}
for i in range(len(df)):
    ptrns = df[df.index == i]['patterns'].values[0]
    rspns = df[df.index == i]['responses'].values[0]
    tag = df[df.index == i]['tag'].values[0]
    for j in range(len(ptrns)):
        dic['tag'].append(tag)
        dic['patterns'].append(ptrns[j])
        dic['responses'].append(rspns)
        
df = pd.DataFrame.from_dict(dic)
df['tag'].unique()

array(['greeting', 'morning', 'afternoon', 'evening', 'night', 'goodbye',
       'thanks', 'neutral-response', 'about', 'skill', 'creation', 'name',
       'help', 'sad', 'stressed', 'worthless', 'depressed', 'happy',
       'casual', 'anxious', 'not-talking', 'sleep', 'scared', 'death',
       'understand', 'done', 'suicide', 'hate-you', 'hate-me', 'default',
       'jokes', 'repeat', 'wrong', 'stupid', 'location', 'something-else',
       'friends', 'ask', 'problem', 'no-approach', 'learn-more',
       'user-agree', 'meditation', 'user-meditation', 'pandora-useful',
       'user-advice', 'learn-mental-health', 'mental-health-fact',
       'fact-1', 'fact-2', 'fact-3', 'fact-5', 'fact-6', 'fact-7',
       'fact-8', 'fact-9', 'fact-10', 'fact-11', 'fact-12', 'fact-13',
       'fact-14', 'fact-15', 'fact-16', 'fact-17', 'fact-18', 'fact-19',
       'fact-20', 'fact-21', 'fact-22', 'fact-23', 'fact-24', 'fact-25',
       'fact-26', 'fact-27', 'fact-28', 'fact-29', 'fact-30', 'fact-31',
 

In [17]:
tokenizer = Tokenizer(lower=True, split=' ')
tokenizer.fit_on_texts(df['patterns'])
tokenizer.get_config()

vacab_size = len(tokenizer.word_index)
print('number of unique words = ', vacab_size)

ptrn2seq = tokenizer.texts_to_sequences(df['patterns'])
X = pad_sequences(ptrn2seq, padding='post')
print('X shape = ', X.shape)

lbl_enc = LabelEncoder()
y = lbl_enc.fit_transform(df['tag'])
print('y shape = ', y.shape)
print('num of classes = ', len(np.unique(y)))

number of unique words =  26
X shape =  (5205, 1)
y shape =  (5205,)
num of classes =  79


In [18]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, LayerNormalization, Dense, Dropout
from tensorflow.keras.utils import plot_model

model = Sequential()
model.add(Input(shape=(X.shape[1])))
model.add(Embedding(input_dim=vacab_size+1, output_dim=100, mask_zero=True))
model.add(LSTM(32, return_sequences=True))
model.add(LayerNormalization())
model.add(LSTM(32, return_sequences=True))
model.add(LayerNormalization())
model.add(LSTM(32))
model.add(LayerNormalization())
model.add(Dense(128, activation="relu"))
model.add(LayerNormalization())
model.add(Dropout(0.2))
model.add(Dense(128, activation="relu"))
model.add(LayerNormalization())
model.add(Dropout(0.2))
model.add(Dense(len(np.unique(y)), activation="softmax"))
model.compile(optimizer='adam', loss="sparse_categorical_crossentropy", metrics=['accuracy'])

In [19]:
model_history = model.fit(x=X,
                          y=y,
                          batch_size=10,
                          callbacks=[tensorflow.keras.callbacks.EarlyStopping(monitor='accuracy', patience=3)],
                          epochs=50)

Epoch 1/50
521/521 [==============================] - 17s 6ms/step - loss: 4.5285 - accuracy: 0.0209
Epoch 2/50
521/521 [==============================] - 3s 6ms/step - loss: 4.3299 - accuracy: 0.0257
Epoch 3/50
521/521 [==============================] - 3s 6ms/step - loss: 4.2738 - accuracy: 0.0321
Epoch 4/50
521/521 [==============================] - 3s 6ms/step - loss: 4.2443 - accuracy: 0.0307
Epoch 5/50
521/521 [==============================] - 3s 6ms/step - loss: 4.2159 - accuracy: 0.0398
Epoch 6/50
521/521 [==============================] - 3s 6ms/step - loss: 4.2025 - accuracy: 0.0340
Epoch 7/50
521/521 [==============================] - 3s 6ms/step - loss: 4.1829 - accuracy: 0.0388
Epoch 8/50
521/521 [==============================] - 3s 6ms/step - loss: 4.1713 - accuracy: 0.0378


In [20]:
# Function to predict intent based on user input
def predict_intent(user_input):
    input_seq = tokenizer.texts_to_sequences([user_input.lower()])
    padded_input_seq = pad_sequences(input_seq, maxlen=max_sequence_length, padding='post')
    predictions = model.predict(padded_input_seq)
    predicted_label = label_tokenizer.inverse_transform([np.argmax(predictions)])
    return predicted_label[0]

# Function to get a response based on the predicted intent
def get_response(predicted_intent):
    for intent in intents:
        if intent['tag'] == predicted_intent:
            responses = intent['responses']
            return np.random.choice(responses)

In [24]:
import tkinter as tk
from tkinter import scrolledtext

# Replace this function with your chatbot logic
def chatbot_response(user_input):
    # Example logic (replace this)
    return "Bot response for: " + user_input

def send_message(event=None):
    user_message = entry_field.get()
    if user_message.strip() != "":
        chat_window.config(state=tk.NORMAL)
        chat_window.insert(tk.END, "You: " + user_message + "\n")
        chat_window.insert(tk.END, "Bot: " + chatbot_response(user_message) + "\n\n")
        chat_window.config(state=tk.DISABLED)
        entry_field.delete(0, tk.END)

# GUI setup
root = tk.Tk()
root.title("Mental Health Chatbot")

chat_window = scrolledtext.ScrolledText(root, width=50, height=20)
chat_window.pack(padx=10, pady=10)
chat_window.config(state=tk.DISABLED)

entry_field = tk.Entry(root, width=40)
entry_field.pack(padx=10, pady=5)
entry_field.bind("<Return>", send_message)

send_button = tk.Button(root, text="Send", command=send_message)
send_button.pack(padx=10, pady=5)

root.mainloop()
